# NSE multi-timeframe swing research

This notebook is deliberately split into numbered cells. It supports one-stock analysis, configurable swing conditions, a conservative daily backtest, and an all-stock NSE screen.

**Data honesty:** free Yahoo data can provide long daily history, but it cannot provide one-minute NSE history from 2010. The notebook reports actual coverage and never presents recent intraday data as a 2010-present test. To research full-history intraday data, point `LICENSED_1M_PATH` at a licensed, locally stored Parquet file.

## Cell 1 — Install libraries

Run once per Jupyter environment, then restart the kernel if requested.

In [1]:
%pip install -q pandas numpy yfinance plotly pyarrow requests tqdm ipywidgets

Note: you may need to restart the kernel to use updated packages.


## Cell 2 — Imports and research configuration

In [2]:
from pathlib import Path
from datetime import date, timedelta
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import requests
import yfinance as yf
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

START_DATE = '2010-01-01'
END_DATE = date.today().isoformat()
SELECTED_SYMBOL = 'HDFCBANK.NS'   # Change this stock here
NSE_TZ = 'Asia/Kolkata'
SESSION_OPEN = '09:15'
SESSION_CLOSE = '15:30'

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'research':
    PROJECT_ROOT = PROJECT_ROOT.parent
RESEARCH_DATA = PROJECT_ROOT / 'research' / 'data'
CACHE_DIR = RESEARCH_DATA / 'notebook_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Optional licensed one-minute file. Required columns:
# timestamp, symbol, open, high, low, close, volume
LICENSED_1M_PATH = PROJECT_ROOT / 'data' / 'licensed_1m' / f'{SELECTED_SYMBOL}.parquet'

print(f'Research window: {START_DATE} to {END_DATE}')
print(f'Selected stock: {SELECTED_SYMBOL}')
print(f'Cache: {CACHE_DIR}')

Research window: 2010-01-01 to 2026-07-24
Selected stock: HDFCBANK.NS
Cache: research\data\notebook_cache


## Cell 3 — Load the NSE stock universe

The official NSE file supplies the **current** equity list. The local 2010 daily research file supplies symbols observed in this project's historical dataset. Neither should be called a survivorship-free point-in-time universe unless delisted securities and historical membership have been independently added.

In [3]:
NSE_EQUITY_URL = 'https://archives.nseindia.com/content/equities/EQUITY_L.csv'
LOCAL_DAILY_FILE = RESEARCH_DATA / 'nse_universe_2010_ohlcv.csv.gz'

def load_current_nse_list():
    headers = {'User-Agent': 'Mozilla/5.0', 'Accept': 'text/csv,*/*'}
    try:
        response = requests.get(NSE_EQUITY_URL, headers=headers, timeout=30)
        response.raise_for_status()
        out = CACHE_DIR / 'nse_current_equities.csv'
        out.write_bytes(response.content)
        current = pd.read_csv(out)
        current.columns = current.columns.str.strip()
        current['SYMBOL'] = current['SYMBOL'].astype(str).str.strip()
        return current
    except Exception as exc:
        print(f'Official NSE list unavailable: {exc}')
        cached = CACHE_DIR / 'nse_current_equities.csv'
        return pd.read_csv(cached) if cached.exists() else pd.DataFrame()

current_nse = load_current_nse_list()

if LOCAL_DAILY_FILE.exists():
    universe_columns = pd.read_csv(
        LOCAL_DAILY_FILE, usecols=['date', 'symbol'], parse_dates=['date']
    )
    historical_coverage = (
        universe_columns.groupby('symbol', observed=True)['date']
        .agg(first_date='min', last_date='max', trading_days='nunique')
        .sort_values(['first_date', 'symbol'])
    )
    historical_symbols = historical_coverage.index.astype(str).tolist()
else:
    historical_coverage = pd.DataFrame()
    historical_symbols = []

current_symbols = (
    (current_nse['SYMBOL'] + '.NS').tolist()
    if not current_nse.empty and 'SYMBOL' in current_nse else []
)

print(f'Current official NSE equity rows: {len(current_symbols):,}')
print(f'Historical daily symbols in local research file: {len(historical_symbols):,}')
display(historical_coverage.head(10))

Current official NSE equity rows: 2,387
Historical daily symbols in local research file: 751


,first_date,last_date,trading_days
symbol,,,
3MINDIA,2010-01-04,2026-07-17,4078
AARTIDRUGS,2010-01-04,2026-07-17,4078
AARTIIND,2010-01-04,2026-07-17,4078
ABB,2010-01-04,2026-07-17,4078
ABBOTINDIA,2010-01-04,2026-07-17,4078
ABREL,2010-01-04,2026-07-17,4078
ACC,2010-01-04,2026-07-17,4078
ACE,2010-01-04,2026-07-17,4078
ADANIENT,2010-01-04,2026-07-17,4078


## Cell 4 — Data-source feasibility matrix

This prevents accidental claims that every timeframe exists from 2010.

In [4]:
requested_frames = pd.DataFrame([
    ('1 minute',  '1m',  'Yahoo: recent only; licensed source needed for 2010-present'),
    ('5 minutes', '5m',  'Resample 1m; Yahoo native history is limited'),
    ('10 minutes','10m', 'Resample 1m or 5m; not a standard Yahoo interval'),
    ('15 minutes','15m', 'Resample 1m; Yahoo native history is limited'),
    ('30 minutes','30m', 'Resample 1m; Yahoo native history is limited'),
    ('60 minutes','60m', 'Resample 1m; Yahoo native history is limited'),
    ('2 hours',   '2h',  'Resample session-anchored 1m; partial final bar'),
    ('5 hours',   '5h',  'Resample session-anchored 1m; 300m + 75m partial bar'),
    ('1 day',     '1d',  'Yahoo/local file can cover 2010-present when listed'),
], columns=['timeframe', 'code', 'honest_availability'])
display(requested_frames)

,timeframe,code,honest_availability
0,1 minute,1m,Yahoo: recent only; licensed source needed for...
1,5 minutes,5m,Resample 1m; Yahoo native history is limited
2,10 minutes,10m,Resample 1m or 5m; not a standard Yahoo interval
3,15 minutes,15m,Resample 1m; Yahoo native history is limited
4,30 minutes,30m,Resample 1m; Yahoo native history is limited
5,60 minutes,60m,Resample 1m; Yahoo native history is limited
6,2 hours,2h,Resample session-anchored 1m; partial final bar
7,5 hours,5h,Resample session-anchored 1m; 300m + 75m parti...
8,1 day,1d,Yahoo/local file can cover 2010-present when l...


## Cell 5 — Get one stock's daily history from 2010 to today

In [5]:
def normalize_ohlcv(df, timestamp_name='timestamp'):
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)
    out.columns = [str(c).strip().lower().replace(' ', '_') for c in out.columns]
    if not isinstance(out.index, pd.DatetimeIndex):
        candidate = next((c for c in ['timestamp', 'datetime', 'date'] if c in out.columns), None)
        if candidate is None:
            raise ValueError('No timestamp/date column found')
        out[candidate] = pd.to_datetime(out[candidate], errors='coerce')
        out = out.set_index(candidate)
    out.index.name = timestamp_name
    required = ['open', 'high', 'low', 'close', 'volume']
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f'Missing OHLCV columns: {missing}')
    out = out[required].apply(pd.to_numeric, errors='coerce')
    out = out.dropna(subset=['open', 'high', 'low', 'close']).sort_index()
    out = out[~out.index.duplicated(keep='last')]
    return out

def load_selected_daily(symbol=SELECTED_SYMBOL):
    plain = symbol.replace('.NS', '')
    if LOCAL_DAILY_FILE.exists():
        raw = pd.read_csv(LOCAL_DAILY_FILE, parse_dates=['date'])
        local = raw.loc[raw['symbol'].astype(str).eq(plain)].copy()
        if not local.empty:
            return normalize_ohlcv(local.set_index('date'), 'date'), 'local research CSV'
    downloaded = yf.download(
        symbol, start=START_DATE, end=(date.today() + timedelta(days=1)).isoformat(),
        interval='1d', auto_adjust=False, progress=False, threads=False
    )
    return normalize_ohlcv(downloaded, 'date'), 'Yahoo Finance'

daily, daily_source = load_selected_daily()
print(f'Daily source: {daily_source}')
print(f'Coverage: {daily.index.min()} to {daily.index.max()} | {len(daily):,} bars')
display(daily.head(3))
display(daily.tail(3))

Daily source: local research CSV
Coverage: 2010-01-04 00:00:00 to 2026-07-17 00:00:00 | 4,078 bars


,open,high,low,close,volume
date,,,,,
2010-01-04,85.00,86.42,84.63,85.28,6100980
2010-01-05,85.50,86.25,85.25,85.36,16773200
2010-01-06,85.70,86.00,84.41,85.42,13279680


,open,high,low,close,volume
date,,,,,
2026-07-15,811.00,819.70,809.70,815.45,21488739
2026-07-16,814.50,818.00,806.50,808.30,18698186
2026-07-17,810.00,823.70,808.50,819.60,17415322


## Cell 6 — Load one-minute data

Priority: licensed local Parquet, otherwise recent Yahoo one-minute data. All timestamps are converted to IST and filtered to the regular NSE session.

In [6]:
def clean_intraday_session(df):
    out = normalize_ohlcv(df)
    if out.index.tz is None:
        out.index = out.index.tz_localize(NSE_TZ)
    else:
        out.index = out.index.tz_convert(NSE_TZ)
    out = out.between_time(SESSION_OPEN, '15:29', inclusive='both')
    out = out[out.index.dayofweek < 5]
    return out

def load_one_minute(symbol=SELECTED_SYMBOL):
    if LICENSED_1M_PATH.exists():
        raw = pd.read_parquet(LICENSED_1M_PATH)
        if 'symbol' in raw.columns:
            raw = raw[raw['symbol'].astype(str).isin([symbol, symbol.replace('.NS', '')])]
        return clean_intraday_session(raw), f'licensed Parquet: {LICENSED_1M_PATH}'

    # Yahoo generally exposes only a short recent window at 1-minute resolution.
    raw = yf.download(
        symbol, period='7d', interval='1m', auto_adjust=False,
        prepost=False, progress=False, threads=False
    )
    return clean_intraday_session(raw), 'Yahoo Finance recent 1-minute window'

one_minute, intraday_source = load_one_minute()
print(f'Intraday source: {intraday_source}')
if one_minute.empty:
    print('No one-minute data returned. Do not continue intraday analysis until data is available.')
else:
    print(f'Coverage: {one_minute.index.min()} to {one_minute.index.max()} | {len(one_minute):,} bars')
    display(one_minute.tail())

Intraday source: Yahoo Finance recent 1-minute window
Coverage: 2026-07-16 09:15:00+05:30 to 2026-07-24 15:29:00+05:30 | 2,625 bars


,open,high,low,close,volume
timestamp,,,,,
2026-07-24 15:25:00+05:30,742.95,742.95,742.65,742.80,177603
2026-07-24 15:26:00+05:30,742.70,742.80,742.30,742.50,172828
2026-07-24 15:27:00+05:30,742.45,742.85,742.40,742.85,315070
2026-07-24 15:28:00+05:30,742.85,743.50,742.75,743.40,120973
2026-07-24 15:29:00+05:30,743.25,743.80,743.00,743.70,107493


## Cell 7 — Session-anchored resampling into every requested timeframe

In [7]:
OHLCV_AGG = {
    'open': 'first', 'high': 'max', 'low': 'min',
    'close': 'last', 'volume': 'sum'
}

FRAME_RULES = {
    '1m': None,
    '5m': '5min',
    '10m': '10min',
    '15m': '15min',
    '30m': '30min',
    '60m': '60min',
    '2h': '120min',
    '5h': '300min',
}

def resample_intraday(df, rule):
    if rule is None:
        return df.copy()
    pieces = []
    for session_date, session in df.groupby(df.index.date):
        # origin='start_day' + offset 09:15 anchors every bar to NSE open.
        bars = session.resample(
            rule, origin='start_day', offset='9h15min',
            label='left', closed='left'
        ).agg(OHLCV_AGG)
        bars['source_minutes'] = session['close'].resample(
            rule, origin='start_day', offset='9h15min',
            label='left', closed='left'
        ).count()
        pieces.append(bars.dropna(subset=['open', 'high', 'low', 'close']))
    return pd.concat(pieces).sort_index() if pieces else pd.DataFrame()

frames = {name: resample_intraday(one_minute, rule) for name, rule in FRAME_RULES.items()}
frames['1d'] = daily.copy()

coverage = []
expected_minutes = {'1m': 1, '5m': 5, '10m': 10, '15m': 15,
                    '30m': 30, '60m': 60, '2h': 120, '5h': 300}
for name, df in frames.items():
    if df.empty:
        coverage.append((name, 0, None, None, 0))
        continue
    partial = 0
    if name in expected_minutes and 'source_minutes' in df:
        partial = int((df['source_minutes'] < expected_minutes[name]).sum())
    coverage.append((name, len(df), df.index.min(), df.index.max(), partial))

coverage_df = pd.DataFrame(
    coverage, columns=['timeframe', 'bars', 'first_timestamp', 'last_timestamp', 'partial_bars']
)
display(coverage_df)

,timeframe,bars,first_timestamp,last_timestamp,partial_bars
0,1m,2625,2026-07-16 09:15:00+05:30,2026-07-24 15:29:00+05:30,0
1,5m,525,2026-07-16 09:15:00+05:30,2026-07-24 15:25:00+05:30,0
2,10m,266,2026-07-16 09:15:00+05:30,2026-07-24 15:25:00+05:30,7
3,15m,175,2026-07-16 09:15:00+05:30,2026-07-24 15:15:00+05:30,0
4,30m,91,2026-07-16 09:15:00+05:30,2026-07-24 15:15:00+05:30,7
5,60m,49,2026-07-16 09:15:00+05:30,2026-07-24 15:15:00+05:30,7
6,2h,28,2026-07-16 09:15:00+05:30,2026-07-24 15:15:00+05:30,7
7,5h,14,2026-07-16 09:15:00+05:30,2026-07-24 14:15:00+05:30,7
8,1d,4078,2010-01-04 00:00:00,2026-07-17 00:00:00,0


## Cell 8 — Indicators without lookahead

In [8]:
def add_indicators(df):
    out = df.copy()
    close = out['close']
    out['ema9'] = close.ewm(span=9, adjust=False).mean()
    out['ema20'] = close.ewm(span=20, adjust=False).mean()
    out['ema50'] = close.ewm(span=50, adjust=False).mean()

    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    out['rsi14'] = 100 - (100 / (1 + rs))

    previous_close = close.shift(1)
    true_range = pd.concat([
        out['high'] - out['low'],
        (out['high'] - previous_close).abs(),
        (out['low'] - previous_close).abs(),
    ], axis=1).max(axis=1)
    out['atr14'] = true_range.ewm(alpha=1/14, adjust=False).mean()
    out['volume_ma20'] = out['volume'].rolling(20).mean()
    out['volume_ratio'] = out['volume'] / out['volume_ma20'].replace(0, np.nan)

    # VWAP resets each trading session for intraday data.
    if isinstance(out.index, pd.DatetimeIndex) and out.index.tz is not None:
        typical = (out['high'] + out['low'] + out['close']) / 3
        session_key = pd.Series(out.index.date, index=out.index)
        cumulative_pv = (typical * out['volume']).groupby(session_key).cumsum()
        cumulative_volume = out['volume'].groupby(session_key).cumsum().replace(0, np.nan)
        out['vwap'] = cumulative_pv / cumulative_volume
    return out

frames = {name: add_indicators(df) if not df.empty else df for name, df in frames.items()}
display(frames['1d'].tail(10))

,open,high,low,close,volume,ema9,ema20,ema50,rsi14,atr14,volume_ma20,volume_ratio
date,,,,,,,,,,,,
2026-07-06,805.00,831.50,805.00,829.85,63203327,800.62,788.43,786.46,69.84,16.22,"36,272,861.60",1.74
2026-07-07,843.00,843.00,826.10,829.30,57611729,806.36,792.32,788.14,69.50,16.27,"38,075,913.05",1.51
2026-07-08,825.90,834.35,806.95,810.30,38754632,807.14,794.03,789.01,58.93,17.06,"38,086,666.50",1.02
2026-07-09,813.00,822.85,812.30,817.55,39891798,809.23,796.27,790.13,61.35,16.74,"37,858,348.45",1.05
2026-07-10,824.00,827.70,820.30,824.95,23595785,812.37,799.00,791.49,63.70,16.27,"37,406,537.35",0.63
2026-07-13,817.00,821.95,811.00,817.95,21741736,813.49,800.81,792.53,59.99,16.10,"36,689,216.20",0.59
2026-07-14,811.40,818.40,807.00,809.40,25605395,812.67,801.63,793.19,55.72,15.77,"35,969,969.25",0.71
2026-07-15,811.00,819.70,809.70,815.45,21488739,813.23,802.94,794.06,57.99,15.38,"35,653,357.70",0.60
2026-07-16,814.50,818.00,806.50,808.30,18698186,812.24,803.45,794.62,54.43,15.10,"34,967,986.80",0.53


## Cell 9 — Visualize one selected timeframe

Change `VIEW_FRAME` and rerun the cell.

In [9]:
VIEW_FRAME = '15m'   # 1m, 5m, 10m, 15m, 30m, 60m, 2h, 5h, 1d
VIEW_BARS = 250

def plot_stock(df, symbol, timeframe, bars=250):
    view = df.tail(bars).copy()
    if view.empty:
        print(f'No data available for {timeframe}')
        return
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.60, 0.20, 0.20], vertical_spacing=0.03,
        subplot_titles=(f'{symbol} — {timeframe}', 'Volume', 'RSI 14')
    )
    fig.add_trace(go.Candlestick(
        x=view.index, open=view.open, high=view.high,
        low=view.low, close=view.close, name='OHLC'
    ), row=1, col=1)
    for column, name, color in [
        ('ema20', 'EMA 20', '#2563eb'), ('ema50', 'EMA 50', '#f59e0b')
    ]:
        fig.add_trace(go.Scatter(
            x=view.index, y=view[column], name=name,
            line=dict(width=1.4, color=color)
        ), row=1, col=1)
    if 'vwap' in view:
        fig.add_trace(go.Scatter(
            x=view.index, y=view['vwap'], name='VWAP',
            line=dict(width=1.2, color='#7c3aed', dash='dot')
        ), row=1, col=1)
    volume_colors = np.where(view.close >= view.open, '#16a34a', '#dc2626')
    fig.add_trace(go.Bar(
        x=view.index, y=view.volume, marker_color=volume_colors, name='Volume'
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=view.index, y=view.rsi14, name='RSI 14', line=dict(color='#0891b2')
    ), row=3, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='#dc2626', row=3, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='#16a34a', row=3, col=1)
    fig.update_layout(
        height=850, template='plotly_white', xaxis_rangeslider_visible=False,
        hovermode='x unified', legend_orientation='h'
    )
    fig.show()

plot_stock(frames[VIEW_FRAME], SELECTED_SYMBOL, VIEW_FRAME, VIEW_BARS)

## Cell 10 — Configurable swing conditions and multi-timeframe status

These are example research conditions, not investment advice. Edit the thresholds and rerun.

In [10]:
CONDITIONS = {
    'minimum_rsi': 50,
    'maximum_rsi': 70,
    'minimum_volume_ratio': 1.20,
    'require_close_above_ema20': True,
    'require_ema20_above_ema50': True,
    'require_intraday_close_above_vwap': True,
}

def condition_mask(df, intraday=False):
    mask = df['rsi14'].between(CONDITIONS['minimum_rsi'], CONDITIONS['maximum_rsi'])
    mask &= df['volume_ratio'].ge(CONDITIONS['minimum_volume_ratio'])
    if CONDITIONS['require_close_above_ema20']:
        mask &= df['close'].gt(df['ema20'])
    if CONDITIONS['require_ema20_above_ema50']:
        mask &= df['ema20'].gt(df['ema50'])
    if intraday and CONDITIONS['require_intraday_close_above_vwap'] and 'vwap' in df:
        mask &= df['close'].gt(df['vwap'])
    return mask.fillna(False)

status_rows = []
for name, df in frames.items():
    if df.empty:
        status_rows.append((name, None, None, None, None, 'NO DATA'))
        continue
    latest = df.iloc[-1]
    passed = bool(condition_mask(df, intraday=name != '1d').iloc[-1])
    status_rows.append((
        name, df.index[-1], latest['close'], latest['rsi14'],
        latest['volume_ratio'], 'PASS' if passed else 'WAIT'
    ))

multi_timeframe_status = pd.DataFrame(status_rows, columns=[
    'timeframe', 'latest_bar', 'close', 'rsi14', 'volume_ratio', 'condition'
])
display(multi_timeframe_status)

,timeframe,latest_bar,close,rsi14,volume_ratio,condition
0,1m,2026-07-24 15:29:00+05:30,743.70,57.52,0.77,WAIT
1,5m,2026-07-24 15:25:00+05:30,743.70,45.74,2.54,WAIT
2,10m,2026-07-24 15:25:00+05:30,743.70,47.70,1.52,WAIT
3,15m,2026-07-24 15:15:00+05:30,743.70,46.59,3.14,WAIT
4,30m,2026-07-24 15:15:00+05:30,743.70,41.79,1.13,WAIT
5,60m,2026-07-24 15:15:00+05:30,743.70,34.21,0.67,WAIT
6,2h,2026-07-24 15:15:00+05:30,743.70,20.40,0.28,WAIT
7,5h,2026-07-24 14:15:00+05:30,743.70,14.60,NaN,WAIT
8,1d,2026-07-17 00:00:00,819.60,58.75,0.52,WAIT


## Cell 11 — Conservative daily swing backtest

The condition is evaluated on a completed daily candle. Entry is the next session's open with adverse slippage. If stop and target are both touched on the same candle, the stop is assumed to occur first.

In [11]:
BACKTEST = {
    'atr_stop_multiple': 2.5,
    'reward_risk_multiple': 2.0,
    'maximum_holding_days': 20,
    'slippage_bps_each_side': 5,
    'round_trip_cost_bps': 20,
}

def backtest_daily_swing(df):
    data = add_indicators(df).dropna(subset=['atr14', 'ema50', 'rsi14']).copy()
    data['signal'] = condition_mask(data, intraday=False)
    trades = []
    i = 1
    while i < len(data):
        if not bool(data['signal'].iloc[i - 1]):
            i += 1
            continue

        signal_time = data.index[i - 1]
        entry_time = data.index[i]
        entry = data['open'].iloc[i] * (1 + BACKTEST['slippage_bps_each_side'] / 10_000)
        atr = data['atr14'].iloc[i - 1]
        stop = entry - BACKTEST['atr_stop_multiple'] * atr
        risk = entry - stop
        target = entry + BACKTEST['reward_risk_multiple'] * risk

        exit_index = min(i + BACKTEST['maximum_holding_days'] - 1, len(data) - 1)
        exit_price = data['close'].iloc[exit_index]
        exit_reason = 'TIME'
        for j in range(i, exit_index + 1):
            # Conservative ordering when daily OHLC cannot reveal which came first.
            if data['low'].iloc[j] <= stop:
                exit_index, exit_price, exit_reason = j, stop, 'STOP'
                break
            if data['high'].iloc[j] >= target:
                exit_index, exit_price, exit_reason = j, target, 'TARGET'
                break

        exit_price *= (1 - BACKTEST['slippage_bps_each_side'] / 10_000)
        gross_return = exit_price / entry - 1
        net_return = gross_return - BACKTEST['round_trip_cost_bps'] / 10_000
        trades.append({
            'signal_time': signal_time, 'entry_time': entry_time,
            'exit_time': data.index[exit_index], 'entry': entry,
            'stop': stop, 'target': target, 'exit': exit_price,
            'exit_reason': exit_reason, 'net_return': net_return,
            'holding_bars': exit_index - i + 1,
        })
        i = exit_index + 1  # no overlapping position in the same stock

    trades = pd.DataFrame(trades)
    if not trades.empty:
        trades['equity'] = (1 + trades['net_return']).cumprod()
    return trades

trades = backtest_daily_swing(daily)
if trades.empty:
    print('No trades matched the current conditions.')
else:
    summary = pd.Series({
        'trades': len(trades),
        'win_rate_pct': (trades.net_return > 0).mean() * 100,
        'average_return_pct': trades.net_return.mean() * 100,
        'median_return_pct': trades.net_return.median() * 100,
        'total_compounded_pct': (trades.equity.iloc[-1] - 1) * 100,
        'average_holding_bars': trades.holding_bars.mean(),
    })
    display(summary.to_frame('value'))
    display(trades.tail(10))
    px.line(trades, x='exit_time', y='equity', title=f'{SELECTED_SYMBOL} — net backtest equity').show()

,value
trades,106.00
win_rate_pct,49.06
average_return_pct,0.47
median_return_pct,-0.14
total_compounded_pct,44.94
average_holding_bars,15.60


,signal_time,entry_time,exit_time,entry,stop,target,exit,exit_reason,net_return,holding_bars,equity
96,2024-10-21,2024-10-22,2024-11-19,865.46,824.98,946.42,870.69,TIME,0.00,20,1.43
97,2024-11-19,2024-11-21,2024-12-18,875.14,830.88,963.66,904.90,TIME,0.03,20,1.47
98,2025-03-21,2025-03-24,2025-04-17,889.94,855.22,959.40,958.92,TARGET,0.08,16,1.59
99,2025-05-08,2025-05-09,2025-06-05,955.53,913.98,"1,038.61",974.31,TIME,0.02,20,1.61
100,2025-06-06,2025-06-09,2025-06-13,998.00,959.46,"1,075.08",958.98,STOP,-0.04,5,1.55
101,2025-06-20,2025-06-23,2025-07-18,974.69,940.25,"1,043.57",978.21,TIME,0.00,20,1.55
102,2025-07-21,2025-07-22,2025-08-19,"1,007.05",972.27,"1,076.61",995.05,TIME,-0.01,20,1.53
103,2025-10-28,2025-10-29,2025-11-26,"1,005.40",969.65,"1,076.91","1,003.40",TIME,-0.00,20,1.52
104,2025-11-27,2025-11-28,2025-12-26,"1,007.50",970.09,"1,082.33",991.60,TIME,-0.02,20,1.50
105,2026-07-06,2026-07-07,2026-07-17,843.42,802.87,924.52,819.19,TIME,-0.03,9,1.45


## Cell 12 — Load all-stock daily data

The existing compressed research dataset is preferred. The optional Yahoo fallback is deliberately capped because downloading every NSE stock in one notebook call is slow and rate-limit sensitive.

In [12]:
MAX_YAHOO_FALLBACK_SYMBOLS = 100

def load_all_stock_daily():
    if LOCAL_DAILY_FILE.exists():
        all_daily = pd.read_csv(LOCAL_DAILY_FILE, parse_dates=['date'])
        all_daily['symbol'] = all_daily['symbol'].astype(str)
        return all_daily.sort_values(['symbol', 'date']), 'local 2010 daily research dataset'

    symbols = current_symbols[:MAX_YAHOO_FALLBACK_SYMBOLS]
    if not symbols:
        raise RuntimeError('No NSE universe available')
    downloaded = yf.download(
        symbols, start=START_DATE, end=(date.today() + timedelta(days=1)).isoformat(),
        auto_adjust=False, group_by='ticker', progress=True, threads=True
    )
    pieces = []
    for symbol in symbols:
        try:
            part = downloaded[symbol].reset_index()
            part.columns = [str(c).lower().replace(' ', '_') for c in part.columns]
            part['symbol'] = symbol.replace('.NS', '')
            pieces.append(part.rename(columns={'date': 'date'}))
        except Exception:
            pass
    return pd.concat(pieces, ignore_index=True), f'Yahoo fallback ({len(symbols)} symbols maximum)'

all_daily, all_daily_source = load_all_stock_daily()
print(f'Source: {all_daily_source}')
print(f'Rows: {len(all_daily):,} | Symbols: {all_daily.symbol.nunique():,}')
print(f'Dates: {all_daily.date.min()} to {all_daily.date.max()}')

Source: local 2010 daily research dataset
Rows: 2,110,297 | Symbols: 751
Dates: 2010-01-04 00:00:00 to 2026-07-17 00:00:00


## Cell 13 — Screen all stocks using completed daily candles

In [13]:
def latest_stock_metrics(group):
    group = group.sort_values('date').copy()
    close = group['close'].astype(float)
    volume = group['volume'].astype(float)
    if len(group) < 65:
        return None
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    rsi = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
    ema20 = close.ewm(span=20, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    return pd.Series({
        'date': group['date'].iloc[-1],
        'close': close.iloc[-1],
        'return_20d_pct': (close.iloc[-1] / close.iloc[-21] - 1) * 100,
        'return_60d_pct': (close.iloc[-1] / close.iloc[-61] - 1) * 100,
        'rsi14': rsi.iloc[-1],
        'ema20': ema20.iloc[-1],
        'ema50': ema50.iloc[-1],
        'volume_ratio': volume.iloc[-1] / volume.tail(20).mean() if volume.tail(20).mean() else np.nan,
        'average_value_20d': (close * volume).tail(20).mean(),
    })

# Retain only the latest 70 observations per symbol before calculating metrics.
recent_all = all_daily.sort_values(['symbol', 'date']).groupby('symbol', observed=True).tail(70)
metric_rows = []
for symbol, group in recent_all.groupby('symbol', observed=True):
    metrics = latest_stock_metrics(group)
    if metrics is not None:
        metrics['symbol'] = symbol
        metric_rows.append(metrics)
market_screen = pd.DataFrame(metric_rows)

market_screen['trend_ok'] = (
    (market_screen.close > market_screen.ema20) &
    (market_screen.ema20 > market_screen.ema50)
)
market_screen['condition_ok'] = (
    market_screen.trend_ok &
    market_screen.rsi14.between(CONDITIONS['minimum_rsi'], CONDITIONS['maximum_rsi']) &
    market_screen.volume_ratio.ge(CONDITIONS['minimum_volume_ratio'])
)
market_screen['research_score'] = (
    market_screen.return_20d_pct.rank(pct=True) * 35 +
    market_screen.return_60d_pct.rank(pct=True) * 35 +
    market_screen.volume_ratio.clip(0, 3).rank(pct=True) * 20 +
    market_screen.trend_ok.astype(int) * 10
)

candidates = market_screen[
    market_screen.condition_ok & market_screen.average_value_20d.ge(10_000_000)
].sort_values('research_score', ascending=False)

print(f'Screened symbols: {len(market_screen):,}')
print(f'Condition matches: {len(candidates):,}')
display(candidates.head(30)[[
    'symbol', 'date', 'close', 'return_20d_pct', 'return_60d_pct',
    'rsi14', 'volume_ratio', 'average_value_20d', 'research_score'
]])

Screened symbols: 750
Condition matches: 41


,symbol,date,close,return_20d_pct,return_60d_pct,rsi14,volume_ratio,average_value_20d,research_score
568,RELAXO,2026-07-17,439.95,19.60,43.73,68.36,8.88,"1,704,287,746.93",97.11
474,NEOGEN,2026-07-17,"2,175.60",12.75,44.22,61.65,1.41,"273,587,198.40",93.41
18,ACMESOLAR,2026-07-17,385.25,16.07,30.04,62.00,1.46,"788,089,467.54",92.64
266,GRANULES,2026-07-17,880.80,12.62,31.91,66.88,1.27,"1,119,584,525.08",91.12
302,HSCL,2026-07-17,741.55,7.42,46.39,68.88,1.42,"8,670,659,578.15",90.04
428,LXCHEM,2026-07-17,171.76,12.11,21.95,66.18,1.62,"230,709,418.96",89.61
528,PNCINFRA,2026-07-17,245.27,13.13,17.58,68.24,2.69,"163,279,776.43",89.05
525,PIRAMALFIN,2026-07-17,"2,144.30",7.30,26.46,55.38,2.88,"1,075,768,773.72",88.67
37,AKUMS,2026-07-17,664.95,10.51,24.45,60.58,1.21,"226,191,351.74",88.02
653,TANLA,2026-07-17,584.55,11.24,19.17,65.54,1.49,"362,459,470.44",87.33


## Cell 14 — All-stock visualizations

In [14]:
plot_data = market_screen.replace([np.inf, -np.inf], np.nan).dropna(
    subset=['return_20d_pct', 'return_60d_pct', 'average_value_20d']
).copy()
plot_data['liquidity_size'] = np.log10(plot_data['average_value_20d'].clip(lower=1))
plot_data['status'] = np.where(plot_data['condition_ok'], 'Condition pass', 'Wait')

fig = px.scatter(
    plot_data, x='return_60d_pct', y='return_20d_pct',
    size='liquidity_size', color='status', hover_name='symbol',
    hover_data=['close', 'rsi14', 'volume_ratio', 'research_score'],
    title='NSE daily momentum map — latest completed data',
    labels={'return_60d_pct': '60-day return (%)', 'return_20d_pct': '20-day return (%)'}
)
fig.add_hline(y=0, line_width=1, line_color='gray')
fig.add_vline(x=0, line_width=1, line_color='gray')
fig.update_layout(template='plotly_white', height=700)
fig.show()

top = candidates.head(25).sort_values('research_score')
px.bar(
    top, x='research_score', y='symbol', orientation='h',
    color='return_20d_pct', color_continuous_scale='RdYlGn',
    title='Top condition-matching stocks — research score only',
    hover_data=['return_60d_pct', 'rsi14', 'volume_ratio']
).update_layout(template='plotly_white', height=750).show()

## Cell 15 — Market breadth visualization without future data

In [15]:
def add_symbol_trend(group):
    group = group.sort_values('date').copy()
    group['ema20'] = group['close'].ewm(span=20, adjust=False).mean()
    group['ema50'] = group['close'].ewm(span=50, adjust=False).mean()
    group['above_ema20'] = group['close'] > group['ema20']
    group['trend_aligned'] = (group['close'] > group['ema20']) & (group['ema20'] > group['ema50'])
    return group[['date', 'symbol', 'above_ema20', 'trend_aligned']]

# Set a shorter window for responsive notebook execution; expand if needed.
BREADTH_START = (pd.Timestamp(END_DATE) - pd.DateOffset(years=5)).date().isoformat()
breadth_source = all_daily[all_daily.date >= BREADTH_START][['date', 'symbol', 'close']].copy()
breadth_parts = [add_symbol_trend(g) for _, g in breadth_source.groupby('symbol', observed=True)]
breadth_rows = pd.concat(breadth_parts, ignore_index=True)
breadth = breadth_rows.groupby('date', observed=True).agg(
    stocks=('symbol', 'nunique'),
    pct_above_ema20=('above_ema20', 'mean'),
    pct_trend_aligned=('trend_aligned', 'mean'),
).reset_index()
breadth[['pct_above_ema20', 'pct_trend_aligned']] *= 100

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=breadth.date, y=breadth.pct_above_ema20,
    name='% above EMA20', line=dict(width=1.5)
))
fig.add_trace(go.Scatter(
    x=breadth.date, y=breadth.pct_trend_aligned,
    name='% close > EMA20 > EMA50', line=dict(width=1.5)
))
fig.update_layout(
    title='NSE market breadth', yaxis_title='Stocks (%)',
    template='plotly_white', hovermode='x unified', height=600
)
fig.show()

## Cell 16 — Export reproducible results

In [16]:
output_dir = RESEARCH_DATA / 'notebook_outputs'
output_dir.mkdir(parents=True, exist_ok=True)

coverage_df.to_csv(output_dir / f'{SELECTED_SYMBOL}_timeframe_coverage.csv', index=False)
multi_timeframe_status.to_csv(output_dir / f'{SELECTED_SYMBOL}_latest_multiframe_status.csv', index=False)
market_screen.to_parquet(output_dir / 'nse_latest_daily_screen.parquet', index=False)
candidates.to_csv(output_dir / 'nse_condition_candidates.csv', index=False)
if not trades.empty:
    trades.to_csv(output_dir / f'{SELECTED_SYMBOL}_daily_backtest_trades.csv', index=False)

print(f'Exported results to: {output_dir.resolve()}')
print('Review coverage_df before interpreting any strategy result.')

Exported results to: research\data\notebook_outputs
Review coverage_df before interpreting any strategy result.
